In [ ]:
!pip install vllm transformers accelerate


In [ ]:
!pip install litellm

  Using cached litellm-1.79.1-py3-none-any.whl.metadata (30 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 23.9 MB/s eta 0:00:00


In [ ]:
!nohup vllm serve Qwen/Qwen3-1.7B \
  --enable-auto-tool-choice \
  --tool-call-parser hermes \
  --reasoning-parser deepseek_r1 \
  > vllm.log &

nohup: redirecting stderr to stdout


In [ ]:
!tail -n 20 vllm.log # to watch last 20 rows og vllm.log, about 2/3 minutes to have the server ready -> Application startup complete


(APIServer pid=1685) INFO 11-08 15:44:36 [launcher.py:42] Route: /v1/responses/{response_id}/cancel, Methods: POST
(APIServer pid=1685) INFO 11-08 15:44:36 [launcher.py:42] Route: /v1/chat/completions, Methods: POST
(APIServer pid=1685) INFO 11-08 15:44:36 [launcher.py:42] Route: /v1/completions, Methods: POST
(APIServer pid=1685) INFO 11-08 15:44:36 [launcher.py:42] Route: /v1/embeddings, Methods: POST
(APIServer pid=1685) INFO 11-08 15:44:36 [launcher.py:42] Route: /pooling, Methods: POST
(APIServer pid=1685) INFO 11-08 15:44:36 [launcher.py:42] Route: /classify, Methods: POST
(APIServer pid=1685) INFO 11-08 15:44:36 [launcher.py:42] Route: /score, Methods: POST
(APIServer pid=1685) INFO 11-08 15:44:36 [launcher.py:42] Route: /v1/score, Methods: POST
(APIServer pid=1685) INFO 11-08 15:44:36 [launcher.py:42] Route: /v1/audio/transcriptions, Methods: POST
(APIServer pid=1685) INFO 11-08 15:44:36 [launcher.py:42] Route: /v1/audio/translations, Methods: POST
(APIServer pid=1685) INFO 11-

In [ ]:
import sqlite3
import pandas as pd

# connect to local SQLite DB
conn = sqlite3.connect("company.db")

def query_db(sql: str):
    """
    Executes an SQL query and returns the result as JSON

    Args:
        sql: The SQL query to execute

    Returns:
        The result of the query as JSON
    """
    try:
        df = pd.read_sql_query(sql, conn)
        return df.to_dict(orient="records")
    except Exception as e:
        return {"error": str(e)}


In [ ]:
# create  a local db

data = {
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "hire_date": ["2020-01-15", "2021-06-20", "2022-03-10"],
    "salary": [50000, 60000, 70000]
}

df = pd.DataFrame(data)
df.to_sql("employees", conn, if_exists="replace", index=False)


3

In [ ]:

def get_function_by_name(name):
    if name == "query_db":
        return query_db

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "query_db",
            "description": "Find SQL query that returns the correct answer of the user question.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": 'The SQL query to execute on the database.',
                    }
                },
                "required": ["sql"],
            },
        },
    }
]
messages = [
    {
        "role": "system",
        "content": "You are an assistant that converts natural language questions into SQL queries. Always respond by calling the tool 'query_db' with the SQL query."
    },
    {
        "role": "user",
        "content": "Write the query to list all employees with a salary higher than 50000."
    }
]



In [ ]:
from litellm import completion

model_name = "hosted_vllm/Qwen/Qwen3-1.7B"

response = completion(
    model=model_name,
    messages=messages,
    temperature=0.7,
    top_p=0.8,
    max_tokens=512,
    repetition_penalty=1.05,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": True,
            "detailed_thinking": True
        }
    },
    api_base = "http://127.0.0.1:8000/v1",
    tools=TOOLS
)

In [ ]:
print(response.choices[0].message) # observing tool_call.name = 'query_db' tool calling is correctly required

Message(content='\n\n', role='assistant', tool_calls=[ChatCompletionMessageToolCall(function=Function(arguments='{"sql": "SELECT * FROM employees WHERE salary > 50000;"}', name='query_db'), id='chatcmpl-tool-56234c1c53cc4bf5a25834b87885bb4b', type='function')], function_call=None, reasoning_content="\nOkay, the user wants to list all employees whose salary is higher than 50000. I need to write an SQL query for that. Let's think about the structure.\n\nFirst, I assume there's a table called employees. The relevant columns would be employee_id and salary. The condition is salary > 50000. \n\nSo the basic SELECT statement would be SELECT * FROM employees WHERE salary > 50000; That should fetch all employees with salaries above 50,000. I don't see any other parameters or tables involved here. The user didn't mention any specific details like department or job title, so I'll stick to the given information. \n\nI should make sure the syntax is correct. Using asterisks (*) selects all columns

In [ ]:
import json

messages.append(response.choices[0].message.model_dump())

if tool_calls := messages[-1].get("tool_calls", None):
    for tool_call in tool_calls:
        call_id: str = tool_call["id"]
        if fn_call := tool_call.get("function"):
            fn_name: str = fn_call["name"]
            fn_args: dict = json.loads(fn_call["arguments"])

            #fn_res: str = json.dumps(get_function_by_name((fn_name)(**fn_args))

            fn_res: str = get_function_by_name(fn_name)(**fn_args)

            messages.append({
                "role": "tool",
                "content": json.dumps({
                    "fn_args": fn_args,
                    "fn_res": fn_res
                }),
                "tool_call_id": call_id,
            })

In [ ]:

response = completion(
    model=model_name,
    messages=messages,
    temperature=0.7,
    top_p=0.8,
    max_tokens=512,
    repetition_penalty=1.05,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": True,
            "detailed_thinking": True
        }
    },
    api_base = "http://127.0.0.1:8000/v1",
    tools=TOOLS
)

In [ ]:
print(response.choices[0].message.content) # output of the query computed on the db



The query executed successfully and returned the following employees with salaries exceeding 50,000:

1. **Bob**  
   - ID: 2  
   - Hire Date: 2021-06-20  
   - Salary: 60,000  

2. **Charlie**  
   - ID: 3  
   - Hire Date: 2022-03-10  
   - Salary: 70,000  

Let me know if you need further analysis!


obs: since the tool calling is already called and we had already the answer if we recall the model it doesn't need to call the tool again. It can simply answer the question.

In [ ]:
response = completion(
    model=model_name,
    messages=messages,
    temperature=0.7,
    top_p=0.8,
    max_tokens=512,
    repetition_penalty=1.05,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": True,
            "detailed_thinking": True
        }
    },
    api_base = "http://127.0.0.1:8000/v1",
    tools=TOOLS
)

In [ ]:
print(response.choices[0].message) # tool_calling = none

Message(content='\n\nThe SQL query to list all employees with a salary higher than 50,000 is:\n\n```sql\nSELECT * FROM employees WHERE salary > 50000;\n```\n\n**Results:**\n1. **Bob** (Salary: $60,000)\n2. **Charlie** (Salary: $70,000)\n\nThis query successfully retrieves the employees meeting the salary criterion. Let me know if you need further modifications!', role='assistant', tool_calls=None, function_call=None, reasoning_content="\nOkay, let me see. The user asked for a query to list all employees with a salary higher than 50000. I generated the SQL statement SELECT * FROM employees WHERE salary > 50000;. Then, when I executed it, the database returned two results: Bob with a salary of 60000 and Charlie with 70000. \n\nNow, the user probably wants to know if the query is correct or if there's anything else needed. Since the query correctly filters employees with salaries above 50000, the response should confirm that the query works as intended. The tool response shows the results